# Obstacle Dataset Builder (Pexels + COCO + Grounding DINO)

Fully automated: downloads images, auto-annotates with Grounding DINO, packages for training.

**Runtime: T4 GPU** | **Time: ~45 min** | **No manual labeling**

After this notebook finishes, download `obstacle_dataset.zip` and upload to `obstacle_detection.ipynb`.

In [ ]:
%pip install -q autodistill autodistill-grounding-dino supervision Pillow tqdm pyyaml requests bing-image-downloader


In [ ]:
import json, os, random, shutil, time, urllib.request, zipfile
from pathlib import Path
import requests, yaml
from tqdm import tqdm

SEED = 42
random.seed(SEED)
WORK = Path('/content/obstacle_build')
WORK.mkdir(exist_ok=True)
OUTPUT_ZIP = Path('/content/obstacle_dataset.zip')
CLASS_NAMES = ['person','vehicle','animal','rock','stump','fence','ditch']
NUM_CLASSES = 7
merged = WORK / 'merged'
out_img = merged / 'images'
out_lbl = merged / 'labels'
out_img.mkdir(parents=True, exist_ok=True)
out_lbl.mkdir(parents=True, exist_ok=True)
RAW_DIR = WORK / 'raw_pexels'
print('Setup OK')

## Step 1: Pexels API - Download images for all rare classes
Paste your API key below.

In [ ]:
# ====== PASTE YOUR PEXELS API KEY HERE ======
PEXELS_API_KEY = ''  # <-- paste between the quotes
# ============================================
assert PEXELS_API_KEY, 'Paste your Pexels API key above!'
print('Pexels API key set')

In [ ]:
def pexels_search(query, per_page=80, pages=2):
    urls = []
    headers = {'Authorization': PEXELS_API_KEY}
    for page in range(1, pages + 1):
        resp = requests.get('https://api.pexels.com/v1/search',
            headers=headers,
            params={'query': query, 'per_page': per_page, 'page': page, 'orientation': 'landscape'},
            timeout=30)
        if resp.status_code != 200:
            print(f'  API error {resp.status_code}')
            break
        data = resp.json()
        for photo in data.get('photos', []):
            urls.append(photo['src']['large'])
        if not data.get('next_page'): break
        time.sleep(0.4)
    return urls

def dl(url, dest):
    try:
        r = requests.get(url, timeout=20)
        if r.status_code == 200 and len(r.content) > 5000:
            dest.write_bytes(r.content)
            return True
    except: pass
    return False
print('Functions ready')

In [ ]:
QUERIES = {
    'rock': ['rock on dirt path','boulder in grass field','large stone ground outdoor',
             'rocks farm road','stone obstacle trail','boulder meadow',
             'rocky ground agriculture','rocks garden ground','pebbles field path',
             'stone wall field'],
    'stump': ['tree stump field','cut tree stump grass','tree stump farm',
              'old stump forest floor','tree stump orchard','wooden stump ground',
              'tree stump path','deforestation stump','stump removal ground',
              'tree trunk cut field'],
    'fence': ['farm fence rural','wooden fence pasture','wire fence agriculture',
              'metal fence farm','fence post field','barbed wire fence',
              'fence gate farm','picket fence rural','chain link fence outdoor',
              'broken fence field'],
    'ditch': ['irrigation ditch farm','drainage ditch field','water channel agriculture',
              'trench farm road','ditch rural road','dry irrigation canal',
              'farm drainage channel','field ditch water','canal irrigation India',
              'drainage channel rural'],
    'person_field': ['person walking farm field','farmer in field',
              'people working agriculture','person rural path',
              'farmer walking crops','worker outdoor field'],
    'animal_field': ['cow in field farm','dog on farm path','sheep grazing field',
              'horse in pasture','goat farm rural','stray dog road rural',
              'cattle grazing','buffalo farm India'],
}

for cls, queries in QUERIES.items():
    d = RAW_DIR / cls
    d.mkdir(parents=True, exist_ok=True)
    if len(list(d.glob('*.jpg'))) > 80:
        print(f'[skip] {cls} already done')
        continue
    print(f'\n[{cls}]')
    all_urls = []
    for q in queries:
        urls = pexels_search(q, per_page=80, pages=2)
        all_urls.extend(urls)
        print(f'  {q}: {len(urls)}')
        time.sleep(0.3)
    seen = set()
    i = len(list(d.glob('*.jpg')))
    for url in all_urls:
        if url in seen: continue
        seen.add(url)
        if dl(url, d / f'{cls}_{i:04d}.jpg'): i += 1
    print(f'  Total: {i} images')
print('\nPexels download done')

## Step 1b: Bing Image Search backup (no API key needed)

In [ ]:
from bing_image_downloader import downloader
BING_Q = {
    'rock': ['rock on dirt road farm','large boulder grass field','rocks agricultural path'],
    'stump': ['tree stump open field','fresh cut tree stump','rotting stump ground level'],
    'fence': ['farm boundary fence India','wire fence paddock rural','old wooden fence field'],
    'ditch': ['irrigation canal India farm','open drain field side','dry ditch agriculture'],
}
BING_DIR = WORK / 'raw_bing'
for cls, qs in BING_Q.items():
    cd = BING_DIR / cls; cd.mkdir(parents=True, exist_ok=True)
    for q in qs:
        try: downloader.download(q, limit=50, output_dir=str(cd), adult_filter_off=False, force_replace=False, timeout=30)
        except Exception as e: print(f'  {e}')
    # Flatten into RAW_DIR
    dst = RAW_DIR / cls; dst.mkdir(exist_ok=True)
    i = len(list(dst.glob('*')))
    for img in cd.rglob('*'):
        if img.suffix.lower() in {'.jpg','.jpeg','.png'}:
            shutil.copy2(img, dst / f'{cls}_bing_{i:04d}{img.suffix}'); i += 1
    print(f'{cls}: {i} total images now')
print('Bing done')

## Step 2: COCO 2017 val for person / vehicle / animal

In [ ]:
COCO_DIR = WORK / 'coco'; COCO_DIR.mkdir(exist_ok=True)
img_zip = COCO_DIR / 'val2017.zip'
ann_zip = COCO_DIR / 'ann.zip'
if not img_zip.exists():
    !wget -q --show-progress -O {img_zip} http://images.cocodataset.org/zips/val2017.zip
if not ann_zip.exists():
    !wget -q --show-progress -O {ann_zip} http://images.cocodataset.org/annotations/annotations_trainval2017.zip
if not (COCO_DIR/'val2017').exists(): !unzip -q {img_zip} -d {COCO_DIR}
if not (COCO_DIR/'annotations').exists(): !unzip -q {ann_zip} -d {COCO_DIR}
print('COCO ready')

In [ ]:
COCO_REMAP = {1:0, 3:1, 6:1, 8:1, 4:1, 17:2, 18:2, 19:2, 20:2, 21:2, 16:2}
with open(COCO_DIR/'annotations'/'instances_val2017.json') as f: coco=json.load(f)
img_info={i['id']:i for i in coco['images']}
img_anns={}
for a in coco['annotations']:
    if a['category_id'] in COCO_REMAP and not a.get('iscrowd',0):
        img_anns.setdefault(a['image_id'],[]).append(a)
ids=list(img_anns.keys()); random.shuffle(ids)
counts=[0,0,0]; copied=0
for img_id in tqdm(ids[:1500], desc='COCO'):
    info=img_info.get(img_id)
    if not info: continue
    src=COCO_DIR/'val2017'/info['file_name']
    if not src.exists(): continue
    w,h=info['width'],info['height']; lines=[]
    for a in img_anns[img_id]:
        c=COCO_REMAP.get(a['category_id'])
        if c is None: continue
        bx,by,bw,bh=a['bbox']
        if bw<=0 or bh<=0: continue
        cx=min(1,max(0,(bx+bw/2)/w)); cy=min(1,max(0,(by+bh/2)/h))
        nw=min(1,max(.001,bw/w)); nh=min(1,max(.001,bh/h))
        lines.append(f'{c} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}'); counts[c]+=1
    if lines:
        stem=f'coco_{img_id:012d}'
        shutil.copy2(src, out_img/f'{stem}{src.suffix}')
        (out_lbl/f'{stem}.txt').write_text('\n'.join(lines)+'\n'); copied+=1
print(f'COCO: {copied} imgs | person:{counts[0]} vehicle:{counts[1]} animal:{counts[2]}')

## Step 3: Auto-annotate with Grounding DINO (zero-shot)

In [ ]:
from autodistill_grounding_dino import GroundingDINO
from autodistill.detection import CaptionOntology
from PIL import Image as PILImage

PROMPTS = {
    'rock': ('rock . boulder . large stone on ground', 3, 0.25),
    'stump': ('tree stump . wooden stump . cut tree trunk on ground', 4, 0.25),
    'fence': ('fence . wire fence . wooden fence . metal fence . farm fence', 5, 0.20),
    'ditch': ('ditch . trench . canal . drainage channel . irrigation ditch', 6, 0.20),
    'person_field': ('person . human . farmer . worker', 0, 0.30),
    'animal_field': ('cow . dog . sheep . horse . goat . animal . cattle', 2, 0.25),
}
for cls,(prompt,cid,thr) in PROMPTS.items():
    sd = RAW_DIR / cls
    if not sd.exists() or not any(sd.iterdir()): print(f'[skip] {cls}'); continue
    exts={'.jpg','.jpeg','.png'}
    imgs=sorted(p for p in sd.iterdir() if p.suffix.lower() in exts)
    print(f'\n[{cls}] {len(imgs)} images, prompt="{prompt}"')
    ont=CaptionOntology({prompt: cls})
    model=GroundingDINO(ontology=ont, box_threshold=thr)
    ct=0
    for ip in tqdm(imgs, desc=cls):
        try:
            with PILImage.open(ip) as im: iw,ih=im.size
            if iw<100 or ih<100: continue
            res=model.predict(str(ip))
            if res is None or len(res.xyxy)==0: continue
            lines=[]
            for b in res.xyxy:
                x1,y1,x2,y2=b
                cx=min(1,max(0,((x1+x2)/2)/iw))
                cy=min(1,max(0,((y1+y2)/2)/ih))
                bw=min(1,max(.001,(x2-x1)/iw))
                bh=min(1,max(.001,(y2-y1)/ih))
                lines.append(f"{cid} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            if lines:
                s=f"auto_{cls}_{ct:05d}"
                shutil.copy2(ip, out_img/f"{s}{ip.suffix}")
                (out_lbl/f"{s}.txt").write_text("\n".join(lines)+"\n")
                ct+=1
        except: continue
    print(f'  -> {ct} annotated for {CLASS_NAMES[cid]}')

## Step 4: Split, validate, package

In [ ]:
final = WORK / 'final'
if final.exists(): shutil.rmtree(final)
exts={'.jpg','.jpeg','.png'}
pairs=[(i, out_lbl/f'{i.stem}.txt') for i in sorted(out_img.iterdir())
       if i.suffix.lower() in exts and (out_lbl/f'{i.stem}.txt').exists()]
random.shuffle(pairs)
n=len(pairs); t=int(.8*n); v=t+int(.1*n)
splits={'train':pairs[:t],'valid':pairs[t:v],'test':pairs[v:]}
for name,items in splits.items():
    (final/name/'images').mkdir(parents=True,exist_ok=True)
    (final/name/'labels').mkdir(parents=True,exist_ok=True)
    for img,lbl in items:
        shutil.copy2(img, final/name/'images'/img.name)
        shutil.copy2(lbl, final/name/'labels'/lbl.name)
    print(f'{name}: {len(items)}')
cfg={'path':'.','train':'train/images','val':'valid/images',
     'test':'test/images','nc':7,'names':CLASS_NAMES}
(final/'dataset.yaml').write_text(yaml.safe_dump(cfg,sort_keys=False))
cc=[0]*7
for sp in ('train','valid','test'):
    for l in (final/sp/'labels').glob('*.txt'):
        for ln in l.read_text().splitlines():
            p=ln.split()
            if len(p)==5: cc[int(p[0])]+=1
print(f'\nTotal: {n} images')
for nm,ct in zip(CLASS_NAMES,cc):
    print(f'  {nm:10s}: {ct:5d} {"OK" if ct>0 else "MISSING!"}')

In [ ]:
if OUTPUT_ZIP.exists(): OUTPUT_ZIP.unlink()
with zipfile.ZipFile(OUTPUT_ZIP,'w',zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(final.rglob('*')):
        if f.is_file(): zf.write(f, f.relative_to(final))
print(f'\nCreated: {OUTPUT_ZIP.name} ({OUTPUT_ZIP.stat().st_size/1e6:.1f} MB)')
print('Next: upload this to obstacle_detection.ipynb')

In [ ]:
from google.colab import files
files.download(str(OUTPUT_ZIP))